# Analysis Main Notebook

이 노트북은 `Analysis` 안의 분석 도구를 한 곳에서 호출합니다. SSH 환경에서도 결과를 볼 수 있도록 그래프는 `trends`, JSON과 최종 표 CSV는 `final` 아래에 저장합니다.

## 0. 공통 설정

In [73]:
from pathlib import Path
import json
import pandas as pd
import sys

# Determine repository root by climbing from the current working directory.
# This makes imports and relative result/output paths work from the notebook
# even if the notebook server starts from a subdirectory.
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / ".git").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from Analysis.analyzer import Analyzer
from Analysis.distance_metrics import nearestStats
from Analysis.statistics import calcStats, printStats, reportCluster
from Analysis.result_io import finalPoints, listBands, loadAlgoRuns
from Analysis.trends import (
    plotConverge,
    saveReport,
    coverSummary,
)

RESULTS_ROOT = REPO_ROOT / "__RESULTS__"
ALGORITHMS = ["ga", "pso", "drl", "greedy"]
MAP_NAME = "seocho.down"
SEED_BAND = None
GRID_M = 5.0
COVERAGE = 45
TARGET_VALUES = (2, 3)

ANALYSIS_ROOT = RESULTS_ROOT / "analysis"
MAP_DIR_NAME = MAP_NAME.replace("/", "_")


def normalizeAlgorithms(value: str | list[str] | tuple[str, ...]) -> list[str]:
    if isinstance(value, str):
        return [value]
    return [str(item) for item in value]


def buildContext(algorithm: str) -> dict[str, object]:
    bands = listBands(
        results_root=RESULTS_ROOT,
        algorithm=algorithm,
        map_name=MAP_NAME,
    )
    selected_band = SEED_BAND if SEED_BAND is not None else (bands[0] if bands else None)
    map_algo_dir = ANALYSIS_ROOT / MAP_DIR_NAME / algorithm
    trends_dir = map_algo_dir / "trends"
    final_dir = map_algo_dir / "final"
    trends_dir.mkdir(parents=True, exist_ok=True)
    final_dir.mkdir(parents=True, exist_ok=True)
    return {
        "algorithm": algorithm,
        "bands": bands,
        "selected_band": selected_band,
        "run_dir": RESULTS_ROOT / algorithm / MAP_NAME,
        "single_run_dir": RESULTS_ROOT / algorithm / MAP_NAME / selected_band
        if selected_band
        else RESULTS_ROOT / algorithm / MAP_NAME,
        "trends_dir": trends_dir,
        "final_dir": final_dir,
        "report_dir": trends_dir,
    }


ALGORITHMS = normalizeAlgorithms(ALGORITHMS)
CONTEXTS = {algorithm: buildContext(algorithm) for algorithm in ALGORITHMS}
pd.DataFrame.from_dict(CONTEXTS, orient="index")[
    ["selected_band", "trends_dir", "final_dir"]
]


,selected_band,trends_dir,final_dir
ga,40-60,/workspace/__RESULTS__/analysis/seocho.down/ga...,/workspace/__RESULTS__/analysis/seocho.down/ga...
pso,40-60,/workspace/__RESULTS__/analysis/seocho.down/ps...,/workspace/__RESULTS__/analysis/seocho.down/ps...
drl,40-60,/workspace/__RESULTS__/analysis/seocho.down/dr...,/workspace/__RESULTS__/analysis/seocho.down/dr...
greedy,40-60,/workspace/__RESULTS__/analysis/seocho.down/gr...,/workspace/__RESULTS__/analysis/seocho.down/gr...


## 1. 결과 파일과 seed band 확인

In [74]:
for context in CONTEXTS.values():
    algorithm = str(context["algorithm"])
    bands = list(context["bands"])
    records = loadAlgoRuns(
        results_root=RESULTS_ROOT,
        algorithm=algorithm,
        map_name=MAP_NAME,
        seed_band=SEED_BAND,
    )
    context["records"] = records
    print(f"[{algorithm}] seed bands: {bands}")
    print(f"[{algorithm}] run count: {len(records)}")

pd.DataFrame(
    {
        algorithm: {
            "bands": len(context["bands"]),
            "runs": len(context["records"]),
            "selected_band": context["selected_band"],
        }
        for algorithm, context in CONTEXTS.items()
    }
).T


[ga] seed bands: ['40-60', '60-80', '80-100', '100-120', '120-140']
[ga] run count: 50
[pso] seed bands: ['40-60', '60-80', '80-100', '100-120']
[pso] run count: 50
[drl] seed bands: ['40-60', '60-80', '80-100', '100-120', '120-140']
[drl] run count: 50
[greedy] seed bands: ['40-60', '60-80', '80-100', '100-120', '120-140']
[greedy] run count: 50


,bands,runs,selected_band
ga,5,50,40-60
pso,4,50,40-60
drl,5,50,40-60
greedy,5,50,40-60


## 2. 단일 run 세대별 추이

`Analyzer`는 하나의 JSON run에 대해 센서 수, 커버리지, fitness 추이를 확인합니다.

In [75]:
for context in CONTEXTS.values():
    algorithm = str(context["algorithm"])
    single_run_dir = Path(context["single_run_dir"])
    trends_dir = Path(context["trends_dir"])
    analyzer = Analyzer(result_root_path=str(single_run_dir), file_path=0)

    analyzer.plot_evolution_trend(
        xtick_step=50,
        include_corners=True,
        figsize=(10, 3),
        dpi=150,
        show=False,
        save_path=trends_dir / "sensor_count.png",
        close=True,
    )
    analyzer.plot_coverage_trend(
        xtick_step=50,
        figsize=(10, 3),
        dpi=150,
        show=False,
        save_path=trends_dir / "coverage.png",
        close=True,
    )
    analyzer.plot_fitness_trend(
        xtick_step=50,
        figsize=(10, 3),
        dpi=150,
        show=False,
        save_path=trends_dir / "fitness.png",
        close=True,
    )
    print(f"[{algorithm}] saved single-run plots to {trends_dir}")


[ga] saved single-run plots to /workspace/__RESULTS__/analysis/seocho.down/ga/trends
[pso] saved single-run plots to /workspace/__RESULTS__/analysis/seocho.down/pso/trends
[drl] saved single-run plots to /workspace/__RESULTS__/analysis/seocho.down/drl/trends
[greedy] saved single-run plots to /workspace/__RESULTS__/analysis/seocho.down/greedy/trends


## 3. 선택 band의 기본 통계

In [76]:
stats_rows = []
for context in CONTEXTS.values():
    algorithm = str(context["algorithm"])
    single_run_dir = Path(context["single_run_dir"])
    print(f"[{algorithm}]")
    printStats(str(single_run_dir))
    stats = calcStats(str(single_run_dir))
    stats_rows.append(
        {
            "algorithm": algorithm,
            "runs": stats[0],
            "coverage_mean": stats[1],
            "coverage_std": stats[2],
            "corner_mean": stats[3],
            "total_mean": stats[5],
            "optimizer_sec_mean": stats[9],
        }
    )

stats_df = pd.DataFrame(stats_rows).set_index("algorithm")
stats_df.round(3)


[ga]
total runs: 10
[coverage] mean ± std: 94.6515 ± 0.8589
[final] corner sensors mean ± std: 4.00 ± 0.00
[final] total sensors mean ± std: 17.00 ± 0.77
[time] corner mean ± std (sec): 0.000 ± 0.000
[time] optimizer mean ± std (sec): 73.993 ± 1.618
[pso]
total runs: 20
[coverage] mean ± std: 97.8364 ± 0.5521
[final] corner sensors mean ± std: 4.00 ± 0.00
[final] total sensors mean ± std: 44.95 ± 0.92
[time] corner mean ± std (sec): 0.000 ± 0.000
[time] optimizer mean ± std (sec): 214.450 ± 3.544
[drl]
total runs: 10
[coverage] mean ± std: 95.8858 ± 0.0000
[final] corner sensors mean ± std: 4.00 ± 0.00
[final] total sensors mean ± std: 20.00 ± 0.00
[time] corner mean ± std (sec): 0.000 ± 0.000
[time] optimizer mean ± std (sec): 10.638 ± 0.170
[greedy]
total runs: 10
[coverage] mean ± std: 93.9497 ± 0.0000
[final] corner sensors mean ± std: 4.00 ± 0.00
[final] total sensors mean ± std: 19.00 ± 0.00
[time] corner mean ± std (sec): 0.000 ± 0.000
[time] optimizer mean ± std (sec): 0.413 ± 

,runs,coverage_mean,coverage_std,corner_mean,total_mean,optimizer_sec_mean
algorithm,,,,,,
ga,10,94.652,0.859,4.0,17.00,73.993
pso,20,97.836,0.552,4.0,44.95,214.450
drl,10,95.886,0.000,4.0,20.00,10.638
greedy,10,93.950,0.000,4.0,19.00,0.413


## 4. 센서 간 거리 분석

In [77]:
distance_rows = []
cluster_rows = []
for context in CONTEXTS.values():
    algorithm = str(context["algorithm"])
    single_run_dir = Path(context["single_run_dir"])
    records = list(context["records"])
    cluster = reportCluster(str(single_run_dir), MAP_NAME, grid_m=GRID_M)
    if cluster is not None:
        cluster["algorithm"] = algorithm
        cluster_rows.append(cluster)
    for path, run in records:
        row = nearestStats(finalPoints(run), grid_m=GRID_M)
        row["algorithm"] = algorithm
        row["run_name"] = run.get("run_name", path.stem)
        distance_rows.append(row)

distance_df = pd.DataFrame(distance_rows).set_index(["algorithm", "run_name"])
cluster_df = pd.DataFrame(cluster_rows).set_index("algorithm")
cluster_df.round(3)


[seocho.down] total runs: 10
[final] 평균 군집거리 mean ± std: 60.081 ± 2.880 m
[final] 군집거리 min / max: 56.277 / 64.746 m
[seocho.down] total runs: 20
[final] 평균 군집거리 mean ± std: 21.266 ± 2.162 m
[final] 군집거리 min / max: 17.826 / 25.524 m
[seocho.down] total runs: 10
[final] 평균 군집거리 mean ± std: 51.985 ± 0.000 m
[final] 군집거리 min / max: 51.985 / 51.985 m
[seocho.down] total runs: 10
[final] 평균 군집거리 mean ± std: 52.336 ± 0.000 m
[final] 군집거리 min / max: 52.336 / 52.336 m


,mean,std,min,max,n_runs
algorithm,,,,,
ga,60.081,2.880,56.277,64.746,10
pso,21.266,2.162,17.826,25.524,20
drl,51.985,0.000,51.985,51.985,10
greedy,52.336,0.000,52.336,52.336,10


## 5. 초기 seed 센서수별 수렴 분석

`metric='best'`는 세대별 best solution 센서수를 기준으로 수렴을 봅니다. `metric='avg'`로 바꾸면 로그의 평균 센서 수 기준으로 분석합니다.

In [78]:
convergence_rows = []
for context in CONTEXTS.values():
    algorithm = str(context["algorithm"])
    bands = list(context["bands"])
    trends_dir = Path(context["trends_dir"])
    convergence = plotConverge(
        results_root=RESULTS_ROOT,
        algorithm=algorithm,
        map_name=MAP_NAME,
        seed_bands=bands,
        include_corners=True,
        metric="avg",
        threshold=0.5,
        dpi=300,
        show=False,
        save_path=trends_dir / "convergence.png",
    )
    convergence_gen = convergence["convergence"]["convergence_gen"]
    convergence_rows.append(
        {
            "algorithm": algorithm,
            "convergence_gen": convergence_gen,
        }
    )
    print(f"[{algorithm}] convergence generation: {convergence_gen}")

convergence_df = pd.DataFrame(convergence_rows).set_index("algorithm")
convergence_df


[ga] convergence generation: 2
[pso] convergence generation: 2
[drl] convergence generation: 100
[greedy] convergence generation: 16


,convergence_gen
algorithm,
ga,2
pso,2
drl,100
greedy,16


## 6. 전체 커버리지와 중첩 커버리지 분석

In [79]:
report_rows = []
for context in CONTEXTS.values():
    algorithm = str(context["algorithm"])
    bands = list(context["bands"])
    report = saveReport(
        results_root=RESULTS_ROOT,
        algorithm=algorithm,
        map_name=MAP_NAME,
        output_dir=Path(context["report_dir"]),
        summary_dir=Path(context["final_dir"]),
        seed_bands=bands,
        include_corners=True,
        metric="best",
        threshold=0.5,
        target_values=TARGET_VALUES,
        dpi=300,
        show=False,
    )
    report_rows.append(
        {
            "algorithm": algorithm,
            **{
                key: report[key]
                for key in [
                    "convergence_plot",
                    "coverage_overlap_plot",
                    "coverage_overlap_summary",
                ]
            },
        }
    )

report_df = pd.DataFrame(report_rows).set_index("algorithm")
report_df


,convergence_plot,coverage_overlap_plot,coverage_overlap_summary
algorithm,,,
ga,/workspace/__RESULTS__/analysis/seocho.down/ga...,/workspace/__RESULTS__/analysis/seocho.down/ga...,/workspace/__RESULTS__/analysis/seocho.down/ga...
pso,/workspace/__RESULTS__/analysis/seocho.down/ps...,/workspace/__RESULTS__/analysis/seocho.down/ps...,/workspace/__RESULTS__/analysis/seocho.down/ps...
drl,/workspace/__RESULTS__/analysis/seocho.down/dr...,/workspace/__RESULTS__/analysis/seocho.down/dr...,/workspace/__RESULTS__/analysis/seocho.down/dr...
greedy,/workspace/__RESULTS__/analysis/seocho.down/gr...,/workspace/__RESULTS__/analysis/seocho.down/gr...,/workspace/__RESULTS__/analysis/seocho.down/gr...


## 7. 논문 표용 요약 CSV

In [80]:
summary_tables = {}
for context in CONTEXTS.values():
    algorithm = str(context["algorithm"])
    bands = list(context["bands"])
    final_dir = Path(context["final_dir"])
    summary = coverSummary(
        results_root=RESULTS_ROOT,
        algorithm=algorithm,
        map_name=MAP_NAME,
        seed_bands=bands,
        target_values=TARGET_VALUES,
    )
    summary_df = pd.DataFrame.from_dict(summary, orient="index")
    csv_path = final_dir / "summary.csv"
    summary_df.to_csv(csv_path, encoding="utf-8-sig")
    summary_tables[algorithm] = summary_df
    print(f"[{algorithm}] saved {csv_path}")

combined_summary = pd.concat(summary_tables, names=["algorithm", "group"])
combined_summary[
    [
        "runs",
        "n_sensors_mean",
        "coverage_percent_mean",
        "overlap_percent_of_target_mean",
        "overlap_percent_of_covered_mean",
        "redundant_hit_percent_mean",
        "logged_final_coverage_mean",
    ]
].round(3)


[ga] saved /workspace/__RESULTS__/analysis/seocho.down/ga/final/summary.csv
[pso] saved /workspace/__RESULTS__/analysis/seocho.down/pso/final/summary.csv
[drl] saved /workspace/__RESULTS__/analysis/seocho.down/drl/final/summary.csv
[greedy] saved /workspace/__RESULTS__/analysis/seocho.down/greedy/final/summary.csv


runs  n_sensors_mean  coverage_percent_mean  \
algorithm group                                                  
ga        40-60    10.0           17.00                 94.768   
          60-80    10.0           17.50                 95.310   
          80-100   10.0           18.00                 95.586   
          100-120  10.0           18.10                 95.591   
          120-140  10.0           17.50                 95.397   
pso       40-60    20.0           44.95                 97.836   
          60-80    10.0           64.90                 97.691   
          80-100   10.0           86.10                 98.083   
          100-120  10.0          106.80                 98.195   
drl       40-60    10.0           20.00                 95.886   
          60-80    10.0           18.00                 93.950   
          80-100   10.0           20.00                 95.886   
          100-120  10.0           20.00                 95.886   
          120-140  10.0           20.00                 95.692   
greedy    all      50.0           19.00                 93.950   

                   overlap_percent_of_target_mean  \
algorithm group                                     
ga        40-60                            32.449   
          60-80                            36.181   
          80-100                           38.117   
          100-120                          38.988   
          120-140                          36.147   
pso       40-60                            88.473   
          60-80                            93.175   
          80-100                           95.891   
          100-120                          95.958   
drl       40-60                            48.935   
          60-80                            46.660   
          80-100                           48.935   
          100-120                          48.935   
          120-140                          50.920   
greedy    all                              49.274   

                   overlap_percent_of_covered_mean  \
algorithm group                                      
ga        40-60                             34.206   
          60-80                             37.944   
          80-100                            39.806   
          100-120                           40.760   
          120-140                           37.833   
pso       40-60                             90.433   
          60-80                             95.373   
          80-100                            97.764   
          100-120                           97.722   
drl       40-60                             51.035   
          60-80                             49.665   
          80-100                            51.035   
          100-120                           51.035   
          120-140                           53.212   
greedy    all                               52.447   

                   redundant_hit_percent_mean  logged_final_coverage_mean  
algorithm group                                                            
ga        40-60                        26.514                      94.768  
          60-80                        28.866                      95.310  
          80-100                       29.640                      95.586  
          100-120                      30.813                      95.591  
          120-140                      28.801                      95.397  
pso       40-60                        70.455                      97.836  
          60-80                        77.644                      97.691  
          80-100                       83.006                      98.083  
          100-120                      86.116                      98.195  
drl       40-60                        36.971                      95.886  
          60-80                        34.909                      93.950  
          80-100                       36.971                      95.886  
          1